### Auto Loder for Customer Data

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.list("eventhub-scope1")

In [0]:
CONNECTION_STR = dbutils.secrets.get(
    scope="eventhub-scope1",
    key="storagekey"
)

In [0]:
spark.conf.set(
  "fs.azure.account.key.project1demo.dfs.core.windows.net",
  CONNECTION_STR
)

In [0]:
from pyspark.sql.functions import *

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", "/Volumes/bankingdata/bronze/schemalocation/customers/")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("badRecordsPath", "/Volumes/bankingdata/bronze/badrecords/customers/")
    .option("pathGlobFilter", "*.csv")
    .load("abfss://project2@project1demo.dfs.core.windows.net/customers/")
)

df = df.withColumn("ingestion_time", current_timestamp()) \
       .withColumn("source_file", input_file_name())

df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/bankingdata/bronze/checkpoints/customers/") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable("bankingdata.bronze.customers")

In [0]:
%sql
select * from bankingdata.bronze.customers